In [1]:
from gerrychain import (GeographicPartition, Partition, Graph, MarkovChain,
                        proposals, updaters, constraints, accept, Election)

from gerrychain.proposals import recom, propose_random_flip

from gerrychain.tree import recursive_tree_part, recursive_seed_part

from gerrychain.metrics import efficiency_gap, mean_median, polsby_popper, partisan_bias

from gerrychain.updaters import cut_edges

from gerrychain.tree import bipartition_tree, find_balanced_edge_cuts_memoization


import geopandas as gpd
import matplotlib.pyplot as plt

import networkx as nx

#from bonsai251029 import *

import numpy as np
import pandas as pd
import csv

In [2]:
graph = Graph.from_json("./GA_Processed/output/GA_Processed_Precincts.json")
df = gpd.read_file("./GA_Processed/output/GA_Processed_Precincts.shp")


In [3]:
def count_spanning(graph):
    laplacian = nx.laplacian_matrix(graph)
    L = np.delete(np.delete(laplacian.todense(), 0, 0), 1, 1)
    return np.linalg.slogdet(L)[1]

def county_splits(partition, df=df):
    df["current"] = df["PRECINCTID"].map(partition.assignment)

    counties = sum(df.groupby("COUNTYFP")['current'].nunique()>1)
    return counties

election_names = [
    "PRE",
    "USS"
]

num_elections = len(election_names)

election_columns = [
  ['G20PRERTRU','G20PREDBID'],
  ['G20USSRPER','G20USSDOSS']
]

my_updaters = {
    "population": updaters.Tally("population", alias="population"),
    "cut_edges": cut_edges,
    "PP":polsby_popper,
    "county_splits": county_splits
}

elections = [
    Election(
        election_names[i],
        {"Democratic": election_columns[i][1], "Republican": election_columns[i][0]},
    )
    for i in range(num_elections)
]

election_updaters = {election.name: election for election in elections}
for node in graph.nodes():
    graph.nodes()[node]["non_NH_Black"] = graph.nodes()[node]["population"] - graph.nodes()[node]["NH_Black"]

my_updaters.update({"NH_Black":Election("NH_Black",{"NH_Black": "NH_Black", "non_NH_Black": "non_NH_Black"})})
my_updaters.update()

# save percentages

my_updaters.update(election_updaters)


In [4]:
import json

with open("./GA_assignment_current.json", "r") as file:
    restart_dict = json.load(file)

In [5]:
restart_partition = GeographicPartition(graph,restart_dict, my_updaters)

In [6]:
# constraints
# Relevant NC metrics: Equal number of representatives (+/- 5% of ideal population)

def county_constraint(partition):
    return partition['county_splits'] < 19

In [7]:
ideal_population = df['population'].sum() / 14 # number of congressional districts
cd_dict = recursive_tree_part(graph, range(14), ideal_population, "population",.02)
tree_partition = GeographicPartition(graph, cd_dict, my_updaters)

In [8]:
from functools import partial

# w/ new county surcharge that weights edges of counties more

county_proposal = partial(
    recom,
    pop_col="population",
    pop_target=ideal_population,
    epsilon=0.01,
    node_repeats=2,
    region_surcharge = {"COUNTYFP":1}, #adds weight of one to the county edges
    method=partial(bipartition_tree,max_attempts=1000,warn_attempts=1000,allow_pair_reselection=True)
)

second_recom_chain = MarkovChain(
    proposal=county_proposal,
    constraints=[],
    accept=accept.always_accept,
    initial_state=tree_partition,
    total_steps=100000
)

In [9]:
cs = []
mms = []
egs = []
pbs =[]
dvp = []
pps = []
bvp = []
mbvp = []
wins = []


temp = 46000
seed = 3
for part in second_recom_chain:

    temp += 1

    if temp %1000 == 0:

        print(3, temp)
        ad = dict(part.assignment)

        with open(f"GA_assignment_current.json", "w") as file:
            json.dump(ad, file)


        plt.figure(figsize=(4,10))
        nx.draw(graph, pos = {x:(graph.nodes()[x]['C_X'],graph.nodes()[x]['C_Y']) for x in graph.nodes()},node_color=[ad[x] for x in graph.nodes()],
            cmap='tab20b',node_size=15)
        plt.savefig(f'./GA_Ensemble_Graphs/network_plot_{seed}_{temp}.png')
        plt.close()

        df['current'] = df["PRECINCTID"].map(ad)
        df.plot(column='current',cmap='tab20b')
        plt.axis('off')
        plt.savefig(f'./GA_Ensemble_Graphs/df_plot_{seed}_{temp}.png')
        plt.close()


        ndf = pd.DataFrame({"CountySplits":cs, "MM":mms, 'EG':egs,'PB':pbs,'DWins':wins,'PP':pps})

        ndf.to_csv(f"./GA_Ensemble_Stats/chain_outputs_{seed}_{temp}.csv")

        with open(f"./GA_Ensemble_Stats/DemPercs_{seed}_{temp}.csv", "w") as tf1:
            writer = csv.writer(tf1, lineterminator="\n")
            writer.writerows(dvp)


        with open(f"./GA_Ensemble_Stats/BlackPercs_{seed}_{temp}.csv", "w") as tf1:
            writer = csv.writer(tf1, lineterminator="\n")
            writer.writerows(bvp)

        cs = []
        mms = []
        egs = []
        pbs =[]
        dvp = []
        pps = []
        bvp = []
        mbvp = []
        wins = []



    cs.append(part['county_splits'])
    mms.append(mean_median(part['PRE']))
    egs.append(efficiency_gap(part['PRE']))
    pbs.append(partisan_bias(part['PRE']))
    dvp.append(sorted(part['PRE'].percents("Democratic")))
    pps.append(sum([1/x for x in polsby_popper(part).values()])/7)
    bvp.append(sorted(part['NH_Black'].percents("NH_Black")))
    mbvp.append(max(bvp[-1]))
    wins.append(part['PRE'].wins("Democratic"))


3 47000
3 48000
3 49000
3 50000
3 51000
3 52000
3 53000
3 54000
3 55000
3 56000
3 57000
3 58000
3 59000
3 60000
3 61000
3 62000
3 63000
3 64000
3 65000
3 66000
3 67000
3 68000
3 69000
3 70000
3 71000
3 72000
3 73000
3 74000
3 75000
3 76000
3 77000
3 78000
3 79000
3 80000
3 81000
3 82000
3 83000
3 84000
3 85000
3 86000
3 87000
3 88000
3 89000
3 90000
3 91000
3 92000
3 93000
3 94000
3 95000
3 96000
3 97000
3 98000
3 99000
3 100000
3 101000
3 102000
3 103000


KeyboardInterrupt: 